In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve().parent
database_path = project_root / "data" / "restaurant_risk.duckdb"
conn = duckdb.connect(str(database_path), read_only = True)
print(f"Project root: {project_root}")
print(f"Database: {database_path}")

Project root: C:\Users\User\projects\restaurant-inspection-prioritization
Database: C:\Users\User\projects\restaurant-inspection-prioritization\data\restaurant_risk.duckdb


## 1. Load the Modeling Dataset

In [2]:
modeling_query = """
SELECT
    f.*,

    l.cutoff_inspection_type,
    l.cutoff_is_cycle_initial,
    l.cutoff_is_cycle_reinspection,

    l.target_inspection_id,
    l.target_inspection_date,
    l.target_inspection_type,
    l.days_to_target,

    l.target_total_violations,
    l.target_critical_violation_count,

    l.target_any_critical,
    l.target_two_or_more_critical,
    l.target_three_or_more_critical,
    l.target_four_or_more_critical,

    l.target_high_severity,
    l.target_label

FROM processed.features AS f
INNER JOIN processed.labels AS l
    ON f.cutoff_inspection_id = l.cutoff_inspection_id
"""

df = conn.execute(modeling_query).df()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

BinderException: Binder Error: Table "l" does not have a column named "target_total_violations"

Candidate bindings: : "target_label"

LINE 14:     l.target_total_violations,
             ^

In [ ]:
df.shape

In [ ]:
df.info()

# 1. Analytical Dataset Validity

In [ ]:
required_columns = [
    "camis",
    "cutoff_inspection_id",
    "cutoff_date",
    "target_inspection_id",
    "target_inspection_date",
    "target_inspection_type",
    "days_to_target",
    "target_critical",
    "history_depth_bucket",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

print("Missing required columns:", missing_columns)

In [ ]:
df["cutoff_date"] = pd.to_datetime(df["cutoff_date"])
df["target_inspection_date"] = pd.to_datetime(df["target_inspection_date"])
print("Rows:", f"{len(df):,}")
print("Unique cutoff inspection IDs:", f"{df['cutoff_inspection_id'].nunique():,}")
print("Duplicate cutoff inspection IDs:", f"{df['cutoff_inspection_id'].duplicated().sum():,}")
print("Null target labels:", f"{df['target_critical'].isna().sum():,}")
print("Targets not strictly after cutoff:", f"{(df['target_inspection_date'] <= df['cutoff_date']).sum():,}")
print("Non-positive target horizons:", f"{(df['days_to_target'] <= 0).sum():,}")

In [ ]:
print("Target value counts:")
display(df["target_critical"].value_counts(dropna=False))

print("\nTarget label values:")
print(sorted(df["target_critical"].dropna().unique()))

# 2. Dataset Card

In [ ]:
dataset_card = pd.DataFrame(
    {
        "metric": [
            "Prediction examples",
            "Unique restaurants",
            "Unique cutoff inspections",
            "Unique target inspections",
            "Earliest cutoff date",
            "Latest cutoff date",
            "Earliest target date",
            "Latest target date",
            "Positive targets",
            "Target prevalence",
            "Median days to target",
        ],
        "value": [
            len(df),
            df["camis"].nunique(),
            df["cutoff_inspection_id"].nunique(),
            df["target_inspection_id"].nunique(),
            df["cutoff_date"].min().date(),
            df["cutoff_date"].max().date(),
            df["target_inspection_date"].min().date(),
            df["target_inspection_date"].max().date(),
            int(df["target_critical"].sum()),
            f"{df['target_critical'].mean():.2%}",
            float(df["days_to_target"].median()),
        ],
    }
)

dataset_card

# 3. Target Inspection Outcome

In [ ]:
target_summary = (
    df["target_high_severity"]
    .agg(
        prediction_examples="size",
        positive_targets="sum",
        target_prevalence="mean",
    )
    .to_frame()
)

target_summary

In [ ]:
target_type_summary = (
    df.groupby("target_inspection_type", dropna=False)["target_critical"]
    .agg(
        examples="size",
        positive_targets="sum",
        target_rate="mean",ddddd
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

target_type_summary

# 4. Target Construction Audit

In [ ]:
inspection_type_audit = conn.execute("""
    SELECT
        inspection_type,
        COUNT(*) AS inspection_events,
        SUM(
            CASE
                WHEN has_critical_violation THEN 1
                ELSE 0
            END
        ) AS critical_inspection_events,
        AVG(
            CASE
                WHEN has_critical_violation THEN 1.0
                ELSE 0.0
            END
        ) AS critical_event_rate,
        AVG(critical_violation_count) AS avg_critical_violations_per_event,
        MAX(critical_violation_count) AS max_critical_violations_per_event
    FROM processed.inspections
    GROUP BY inspection_type
    ORDER BY inspection_events DESC
""").df()

inspection_type_audit

In [ ]:
critical_flag_audit = conn.execute("""
    SELECT
        critical_flag,
        COUNT(*) AS violation_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_of_rows
    FROM processed.deduplicated_inspection_rows
    GROUP BY critical_flag
    ORDER BY violation_rows DESC
""").df()

critical_flag_audit

In [ ]:
critical_count_distribution = conn.execute("""
    SELECT
        critical_violation_count,
        COUNT(*) AS inspection_events,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_of_events
    FROM processed.inspections
    WHERE inspection_type = 'Cycle Inspection / Initial Inspection'
    GROUP BY critical_violation_count
    ORDER BY critical_violation_count
""").df()

critical_count_distribution

In [ ]:
example_critical_event = conn.execute("""
    SELECT
        inspection_id,
        camis,
        inspection_date,
        inspection_type,
        total_violations,
        critical_violation_count,
        noncritical_violation_count,
        has_critical_violation
    FROM processed.inspections
    WHERE inspection_type = 'Cycle Inspection / Initial Inspection'
      AND has_critical_violation = TRUE
    ORDER BY critical_violation_count DESC, inspection_date DESC
    LIMIT 5
""").df()

example_critical_event

# 5. Target Severity Design

In [ ]:
target_severity_distribution = conn.execute("""
    SELECT
        critical_violation_count,
        COUNT(*) AS inspection_events,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_of_cycle_initial_events,

        SUM(COUNT(*)) OVER (
            ORDER BY critical_violation_count DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS events_at_or_above_threshold,

        ROUND(
            100.0
            * SUM(COUNT(*)) OVER (
                ORDER BY critical_violation_count DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )
            / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_at_or_above_threshold

    FROM processed.inspections
    WHERE inspection_type = 'Cycle Inspection / Initial Inspection'
    GROUP BY critical_violation_count
    ORDER BY critical_violation_count DESC
""").df()

target_severity_distribution

In [ ]:
severity_thresholds = conn.execute("""
    SELECT
        COUNT(*) AS cycle_initial_events,

        SUM(
            CASE WHEN critical_violation_count >= 1 THEN 1 ELSE 0 END
        ) AS positive_at_1,

        SUM(
            CASE WHEN critical_violation_count >= 2 THEN 1 ELSE 0 END
        ) AS positive_at_2,

        SUM(
            CASE WHEN critical_violation_count >= 3 THEN 1 ELSE 0 END
        ) AS positive_at_3,

        SUM(
            CASE WHEN critical_violation_count >= 4 THEN 1 ELSE 0 END
        ) AS positive_at_4

    FROM processed.inspections
    WHERE inspection_type = 'Cycle Inspection / Initial Inspection'
""").df()

severity_thresholds

In [ ]:
total_events = severity_thresholds.loc[
    0,
    "cycle_initial_events",
]

threshold_comparison = pd.DataFrame(
    {
        "threshold": [
            ">= 1 critical violation",
            ">= 2 critical violations",
            ">= 3 critical violations",
            ">= 4 critical violations",
        ],
        "positive_events": [
            severity_thresholds.loc[0, "positive_at_1"],
            severity_thresholds.loc[0, "positive_at_2"],
            severity_thresholds.loc[0, "positive_at_3"],
            severity_thresholds.loc[0, "positive_at_4"],
        ],
    }
)

threshold_comparison["positive_rate"] = (
    threshold_comparison["positive_events"] / total_events
)

threshold_comparison